# Silver Layer — labevents

## READINESS: profiled, rules drafted, awaiting team approval

The profiling cells below were run (results recorded in **Profiling Findings**) and
the Silver rules derived from them are implemented. Until the `profiling_approved`
widget is set to `true`, the publish step raises an error and nothing is written.
The dependency-ordered runner excludes this notebook by default.

## Source
`<source_schema>.labevents` (default `cdr_mimiciv.bronze.labevents`)

## Target
`<target_schema>.labevents` (default `cdr_mimiciv.silver.labevents`), published with
CREATE OR REPLACE semantics. Downstream Gold depends on it. No tables are dropped.

## Grain (official MIMIC-IV hosp documentation, `labevents`)
One row per laboratory measurement, identified by `labevent_id`. Documented fields:
`labevent_id`, `subject_id`, `hadm_id` (nullable: documented as NULL when the observation
falls outside a `transfers` row and has no ED stay identifier),
`specimen_id`, `itemid` (links to `d_labitems`), `order_provider_id`, `charttime`
(usually the time the specimen was acquired), `storetime` (time the measurement was made
available in the laboratory system), `value` (result text), `valuenum` (documented as the
value cast to a numeric type if it is numeric), `valueuom`, `ref_range_lower`,
`ref_range_upper` (values outside the reference range are considered abnormal), `flag`
(documented only as a brief string mainly indicating an abnormal result; its value set is
not enumerated in the documentation), `priority` (documented as routine or stat) and
`comments` (free text; a fully de-identified comment is `___`, a NULL comment means no
comment was made).
Documented NOT NULL: `labevent_id`, `subject_id`, `specimen_id`, `itemid`.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions: preserve source value, identify anomaly, add a derived
column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order.
- Add derived columns (our engineering choice): `value_comparator`,
  `value_censored_num`, `valueuom_std`.
- Add DQ flags (our engineering choice): `dq_result_not_reported`,
  `dq_storetime_before_charttime`, `dq_hadm_not_in_admissions`.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no unit conversion,
  no row removal. `valuenum` is never overwritten.
- `hadm_id` is not imputed: NULL is documented behaviour. Any time-based link to an
  admission is an analytical choice for Gold.

## Open profiling questions
1. Is `labevent_id` unique and never NULL? Are there exact duplicate rows?
2. Missingness of every column, in particular `hadm_id`, `storetime`, `value`,
   `valuenum`, `valueuom`, `ref_range_*`, `flag`, `priority`, `comments`.
3. Which text values occur in `value` where `valuenum` is NULL, and is there any row
   with a numeric `value` but NULL `valuenum` (or the reverse)?
4. Values of `flag`, `priority` and `valueuom` (case or whitespace variants; which values does `flag` take, since the documentation does not enumerate them)?
5. Timestamp order: `storetime` earlier than `charttime`; NULL `charttime`.
6. `ref_range_lower` greater than `ref_range_upper`.
7. Relationships (observational only): `itemid` in `d_labitems`; `subject_id` in
   patients; `(subject_id, hadm_id)` in admissions.
8. The earlier profiling query (`valuenum = 6.1`, `ref_range_upper = 4`, `ng/mL`) is
   retained below to examine a suspected unit or reference-range inconsistency.

## Cost note
`labevents` is very large. Profiling
uses a single aggregate pass where possible, avoids `exceptAll`, and only drills down
into duplicates if the aggregate shows any.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.labevents"
TARGET_TABLE = f"{SILVER}.labevents"
KEY = "labevent_id"

DOCUMENTED_COLUMNS = [
    "labevent_id", "subject_id", "hadm_id", "specimen_id", "itemid",
    "order_provider_id", "charttime", "storetime", "value", "valuenum",
    "valueuom", "ref_range_lower", "ref_range_upper", "flag", "priority", "comments"
]

In [0]:
labevents_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = labevents_bronze.columns

labevents_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
display(labevents_bronze.limit(10))

root
 |-- labevent_id: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- specimen_id: integer (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- value: string (nullable = true)
 |-- valuenum: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- ref_range_lower: double (nullable = true)
 |-- ref_range_upper: double (nullable = true)
 |-- flag: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- comments: string (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []


labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
1,10000032,null,2704548,50931,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T15:56:00.000Z,___,95.0,mg/dL,70.0,100.0,null,ROUTINE,"IF FASTING, 70-100 NORMAL, >125 PROVISIONAL DIABETES."
2,10000032,null,36092842,51071,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,null
3,10000032,null,36092842,51074,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,null
4,10000032,null,36092842,51075,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,"BENZODIAZEPINE IMMUNOASSAY SCREEN DOES NOT DETECT SOME DRUGS,;INCLUDING LORAZEPAM, CLONAZEPAM, AND FLUNITRAZEPAM."
5,10000032,null,36092842,51079,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,null
6,10000032,null,36092842,51087,P69FQC,2180-03-23T11:51:00.000Z,null,null,null,null,null,null,null,ROUTINE,RANDOM.
7,10000032,null,36092842,51089,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:15:00.000Z,null,null,null,null,null,null,ROUTINE,PRESUMPTIVELY POSITIVE.
8,10000032,null,36092842,51090,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,METHADONE ASSAY DETECTS ONLY METHADONE (NOT OTHER OPIATES/OPIOIDS).
9,10000032,null,36092842,51092,P69FQC,2180-03-23T11:51:00.000Z,2180-03-23T16:00:00.000Z,NEG,null,null,null,null,null,ROUTINE,"OPIATE IMMUNOASSAY SCREEN DOES NOT DETECT SYNTHETIC OPIOIDS;SUCH AS METHADONE, OXYCODONE, FENTANYL, BUPRENORPHINE, TRAMADOL,;NALOXONE, MEPERIDINE. SEE ONLINE LAB MANUAL FOR DETAILS."
10,10000032,null,83925500,50853,P69FQC,2180-03-23T11:51:00.000Z,2180-03-25T11:06:00.000Z,___,15.0,ng/mL,30.0,60.0,abnormal,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ACCURATELY.


In [0]:
# single-pass profile: grain/key, missingness of every column, value consistency,
# timestamp order and reference-range order (no repeated scans, no shuffles)
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

profile_exprs = [
    F.count("*").alias("total_rows"),
    F.countDistinct(KEY).alias("distinct_labevent_id"),
]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
if {"value", "valuenum"} <= set(SOURCE_COLUMNS):
    profile_exprs += [
        _cnt(F.col("valuenum").isNull() & F.col("value").isNotNull()).alias("valuenum_null_value_present"),
        _cnt(F.col("valuenum").isNotNull() & F.col("value").isNull()).alias("valuenum_present_value_null"),
    ]
if {"charttime", "storetime"} <= set(SOURCE_COLUMNS):
    profile_exprs += [
        _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
        _cnt(F.col("storetime") == F.col("charttime")).alias("storetime_equals_charttime"),
    ]
if {"ref_range_lower", "ref_range_upper"} <= set(SOURCE_COLUMNS):
    profile_exprs += [
        _cnt(F.col("ref_range_lower") > F.col("ref_range_upper")).alias("ref_lower_gt_upper"),
    ]

lab_profile = labevents_bronze.agg(*profile_exprs).first().asDict()
for k, v in lab_profile.items():
    print(f"{k}: {v}")

total_rows: 158374764
distinct_labevent_id: 158374764
null_labevent_id: 0
null_subject_id: 0
null_hadm_id: 73768897
null_specimen_id: 0
null_itemid: 0
null_order_provider_id: 113933995
null_charttime: 0
null_storetime: 2568511
null_value: 16062783
null_valuenum: 21490341
null_valueuom: 23522996
null_ref_range_lower: 31579618
null_ref_range_upper: 31579618
null_flag: 112405302
null_priority: 7652745
null_comments: 130357314
valuenum_null_value_present: 5427558
valuenum_present_value_null: 0
storetime_before_charttime: 1372
storetime_equals_charttime: 194152
ref_lower_gt_upper: 0


In [0]:
# duplicates: derived from the aggregate; drill down only if the key is not unique
non_null_ids = lab_profile["total_rows"] - lab_profile[f"null_{KEY}"]
print("labevent_id duplicates present:", lab_profile["distinct_labevent_id"] != non_null_ids)

if lab_profile["distinct_labevent_id"] != non_null_ids:
    duplicate_ids = (
        labevents_bronze.groupBy(KEY).count().filter(F.col("count") > 1)
    )
    print("Duplicate labevent_id values:", duplicate_ids.count())
    display(
        labevents_bronze.join(duplicate_ids.limit(50).select(KEY), KEY, "inner")
        .orderBy(KEY)
    )

labevent_id duplicates present: False


In [0]:
# categorical values (small result sets)
for c in ["flag", "priority"]:
    if c in SOURCE_COLUMNS:
        print(f"=== {c} ===")
        display(labevents_bronze.groupBy(c).count().orderBy(F.desc("count")))

if "valueuom" in SOURCE_COLUMNS:
    print("=== valueuom (top 100) ===")
    display(labevents_bronze.groupBy("valueuom").count().orderBy(F.desc("count")).limit(100))

=== flag ===


flag,count
null,112405302
abnormal,45969462


=== priority ===


priority,count
STAT,81500430
ROUTINE,69221589
null,7652745


=== valueuom (top 100) ===


valueuom,count
mg/dL,30028412
null,23522996
mEq/L,22398584
%,21532112
K/uL,12679219
g/dL,8033391
IU/L,6643893
fL,6403010
m/uL,4190936
pg,4152120


In [0]:
# earlier profiling question: text values present where valuenum is NULL
display(
    labevents_bronze
    .where(F.col("valuenum").isNull())
    .groupBy("value")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
)

value,count
null,16062783
NEG,1299393
ART.,534875
NONE,502221
___,451122
HOLD. DISCARD GREATER THAN 4 HOURS OLD.,395912
HOLD. DISCARD GREATER THAN 24 HRS OLD.,340030
VEN.,218440
HOLD.,200333
INTUBATED.,145281


In [0]:
# earlier profiling question: sample joined to the lab dictionary (observational)
labitems_ref = spark.table(f"{BRONZE}.d_labitems")

display(
    labevents_bronze.alias("l")
    .join(labitems_ref.alias("d"), F.col("l.itemid") == F.col("d.itemid"), "left")
    .select(
        "l.labevent_id", "l.itemid", F.col("d.label").alias("lab_test"), "d.fluid",
        "l.value", "l.valuenum", "l.valueuom",
        "l.ref_range_lower", "l.ref_range_upper", "l.flag"
    )
    .limit(30)
)

labevent_id,itemid,lab_test,fluid,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag
35789445,51265,Platelet Count,Blood,292,292.0,K/uL,150.0,400.0,null
35789446,51277,RDW,Blood,14.1,14.1,%,10.5,15.5,null
35789447,51279,Red Blood Cells,Blood,3.76,3.76,m/uL,4.6,6.1,abnormal
35789448,51301,White Blood Cells,Blood,7.1,7.1,K/uL,4.0,10.0,null
35789449,52172,RDW-SD,Blood,53.1,53.1,fL,35.1,46.3,abnormal
35789450,50861,Alanine Aminotransferase (ALT),Blood,18,18.0,IU/L,0.0,40.0,null
35789451,50862,Albumin,Blood,4.5,4.5,g/dL,3.5,5.2,null
35789452,50863,Alkaline Phosphatase,Blood,171,171.0,IU/L,40.0,130.0,abnormal
35789453,50868,Anion Gap,Blood,___,11.0,mEq/L,10.0,18.0,null
35789454,50878,Asparate Aminotransferase (AST),Blood,24,24.0,IU/L,0.0,40.0,null


In [0]:
# earlier profiling question: suspected unit / reference-range inconsistency
display(
    labevents_bronze.alias("l")
    .join(labitems_ref.alias("d"), F.col("l.itemid") == F.col("d.itemid"), "left")
    .where(
        (F.col("l.valuenum") == 6.1)
        & (F.col("l.ref_range_upper") == 4)
        & (F.lower(F.trim(F.col("l.valueuom"))) == "ng/ml")
    )
    .select(
        "l.labevent_id", "l.itemid", F.col("d.label").alias("lab_test"),
        "l.value", "l.valuenum", "l.valueuom",
        "l.ref_range_lower", "l.ref_range_upper", "l.flag"
    )
    .limit(50)
)

labevent_id,itemid,lab_test,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag
36220955,50974,Prostate Specific Antigen,___,6.1,ng/mL,0.0,4.0,abnormal
36311876,50900,Carcinoembyronic Antigen (CEA),___,6.1,ng/mL,0.0,4.0,abnormal
36562766,50900,Carcinoembyronic Antigen (CEA),___,6.1,ng/mL,0.0,4.0,abnormal
37070156,50974,Prostate Specific Antigen,___,6.1,ng/mL,0.0,4.0,abnormal
37237719,50900,Carcinoembyronic Antigen (CEA),___,6.1,ng/mL,0.0,4.0,abnormal
37275734,50900,Carcinoembyronic Antigen (CEA),___,6.1,ng/mL,0.0,4.0,abnormal
39205691,50974,Prostate Specific Antigen,___,6.1,ng/mL,0.0,4.0,abnormal
39682492,50974,Prostate Specific Antigen,___,6.1,ng/mL,0.0,4.0,abnormal
41471815,50974,Prostate Specific Antigen,___,6.1,ng/mL,0.0,4.0,abnormal
42186482,50900,Carcinoembyronic Antigen (CEA),___,6.1,ng/mL,0.0,4.0,abnormal


In [0]:
# relationships (observational only; findings are recorded, not turned into flags here)
# distinct keys are taken first so the large table is scanned once per relationship
itemid_ref = labitems_ref.select("itemid").distinct()
print(
    "distinct labevents itemid not found in d_labitems:",
    labevents_bronze.select("itemid").distinct().join(itemid_ref, "itemid", "left_anti").count()
)

patients_table = f"{SILVER}.patients"
admissions_table = f"{SILVER}.admissions"

if spark.catalog.tableExists(patients_table):
    patients_ref = spark.table(patients_table).select("subject_id").distinct()
    print(
        "distinct labevents subject_id not found in patients:",
        labevents_bronze.select("subject_id").distinct().join(patients_ref, "subject_id", "left_anti").count()
    )
else:
    print(f"{patients_table} not available; subject_id relationship not checked")

if spark.catalog.tableExists(admissions_table):
    admissions_ref = spark.table(admissions_table).select("subject_id", "hadm_id").distinct()
    print(
        "distinct (subject_id, hadm_id) with non-null hadm_id not found in admissions:",
        labevents_bronze.where(F.col("hadm_id").isNotNull())
        .select("subject_id", "hadm_id").distinct()
        .join(admissions_ref, ["subject_id", "hadm_id"], "left_anti").count()
    )
else:
    print(f"{admissions_table} not available; hadm_id relationship not checked")

distinct labevents itemid not found in d_labitems: 0
distinct labevents subject_id not found in patients: 0
distinct (subject_id, hadm_id) with non-null hadm_id not found in admissions: 1


## Profiling Findings

Recorded from the profiling run of 2026-10-19 on `cdr_mimiciv.bronze.labevents`.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows / distinct `labevent_id` | 158,374,764 / 158,374,764 | Key unique; no deduplication |
| NULL `labevent_id`, `subject_id`, `specimen_id`, `itemid`, `charttime` | 0 | None |
| NULL `hadm_id` | 73,768,897 (46.6%) | Keep NULL; [DOC] specimen outside a hospital stay |
| NULL `order_provider_id` | 113,933,995 (71.9%) | Keep |
| NULL `storetime` | 2,568,511 (1.6%) | Keep; documented, no flag |
| NULL `value` / `valuenum` / `valueuom` | 16,062,783 / 21,490,341 / 23,522,996 | Keep |
| NULL `ref_range_lower` / `ref_range_upper` | 31,579,618 each (19.9%) | Keep |
| NULL `flag` / `priority` / `comments` | 112,405,302 / 7,652,745 / 130,357,314 | Keep |
| `valuenum` NULL but `value` present | 5,427,558 | Text results kept; censored numerics parsed into `value_comparator` / `value_censored_num`; failed results flagged `dq_result_not_reported` [OURS] |
| `valuenum` present but `value` NULL | 0 | None |
| `flag` values | `abnormal` 45,969,462; NULL | No change; NULL is not "normal" [DOC: flag "mainly" abnormal, domain not closed] |
| `priority` values | STAT 81,500,430; ROUTINE 69,221,589; NULL | No change |
| `valueuom` | 60 distinct; `N/A` 3,032,870; empty string 845,181 | `valueuom_std`: trimmed, empty string to NULL; spelling variants not merged until checked per itemid [OURS] |
| `storetime` before `charttime` | 1,372 | `dq_storetime_before_charttime` [OURS]; no correction |
| `storetime` equals `charttime` | 194,152 | None |
| `ref_range_lower` > `ref_range_upper` | 0 | None |
| `itemid` not in `d_labitems` | 0 distinct | None |
| `subject_id` not in patients | 0 distinct | None |
| `(subject_id, hadm_id)` not in admissions | 1 distinct pair | `dq_hadm_not_in_admissions` [OURS]; row kept |
| 6.1 ng/mL vs upper 4 (CEA, PSA) | All flagged `abnormal` | Correctly flagged high results, not a unit issue; no action |

In [0]:
# Silver transformation: all source columns unchanged + derived columns + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed

# censored numeric results, e.g. ">50", "<=1.005"; only where valuenum is NULL
CENSORED_PATTERN = r"^(<=|>=|<|>)\s*(-?[0-9]+(\.[0-9]+)?)$"

# failed / not-reported results observed in profiling (explicit list, auditable)
NOT_REPORTED_VALUES = [
    "ERROR", "CANCEL", "CANCELLED", "CANCELED", "UNABLE", "UNABLE TO REPORT",
    "UNABLE TO REPORT,QNS", "UNABLE TO PERFORM", "UNABLE TO RESULT", "NOT DONE"
]

value_trim = F.trim(F.col("value"))
is_censored = F.col("valuenum").isNull() & value_trim.rlike(CENSORED_PATTERN)

# distinct admission keys from Silver admissions (small; drives dq_hadm_not_in_admissions)
admissions_keys = (
    spark.table(f"{SILVER}.admissions")
    .select("subject_id", "hadm_id").distinct()
    .withColumn("_adm_match", F.lit(True))
)

labevents_silver = (
    labevents_bronze.select(*SOURCE_COLUMNS)
    .join(F.broadcast(admissions_keys), ["subject_id", "hadm_id"], "left")
    .select(
        *SOURCE_COLUMNS,
        F.when(is_censored, F.regexp_extract(value_trim, CENSORED_PATTERN, 1))
            .alias("value_comparator"),
        F.when(is_censored, F.regexp_extract(value_trim, CENSORED_PATTERN, 2).cast("double"))
            .alias("value_censored_num"),
        F.when(F.trim(F.col("valueuom")) == "", F.lit(None).cast("string"))
            .otherwise(F.trim(F.col("valueuom")))
            .alias("valueuom_std"),
        F.coalesce(F.upper(value_trim).isin(NOT_REPORTED_VALUES), F.lit(False))
            .alias("dq_result_not_reported"),
        F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False))
            .alias("dq_storetime_before_charttime"),
        (F.col("hadm_id").isNotNull() & F.col("_adm_match").isNull())
            .alias("dq_hadm_not_in_admissions"),
        F.current_timestamp().alias("_silver_processed_at"),
    )
)

expected_schema = {f.name: f.dataType.simpleString() for f in labevents_bronze.schema.fields}
expected_schema.update({
    "value_comparator": "string",
    "value_censored_num": "double",
    "valueuom_std": "string",
    "dq_result_not_reported": "boolean",
    "dq_storetime_before_charttime": "boolean",
    "dq_hadm_not_in_admissions": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (2026-10-19 run)
PROFILED_COUNTS = {
    "dq_storetime_before_charttime": 1372,
}

labevents_silver.printSchema()

root
 |-- labevent_id: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- specimen_id: integer (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- value: string (nullable = true)
 |-- valuenum: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- ref_range_lower: double (nullable = true)
 |-- ref_range_upper: double (nullable = true)
 |-- flag: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- comments: string (nullable = true)
 |-- value_comparator: string (nullable = true)
 |-- value_censored_num: double (nullable = true)
 |-- valueuom_std: string (nullable = true)
 |-- dq_result_not_reported: boolean (nullable = false)
 |-- dq_storetime_before_charttime: boolean (nullable = false)
 |-- dq_hadm_not_in_admissions: boolean (nullable = false)
 

In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "labevents is not approved for publication: run and review the profiling cells, "
        "add the findings and any justified rules, then set widget profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in labevents_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"
assert KEY in pre_schema, f"Key column {KEY} missing"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    labevents_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.labevents


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze
# (no exceptAll on a table of this size)
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct(KEY).alias("distinct_key"),
    F.sum(F.when(F.col(KEY).isNull(), 1).otherwise(0)).alias("null_key"),
    F.sum(F.when(F.col("_silver_processed_at").isNull(), 1).otherwise(0)).alias("null_processed_at"),
    F.sum(F.when(F.col("dq_result_not_reported"), 1).otherwise(0)).alias("dq_result_not_reported"),
    F.sum(F.when(F.col("dq_storetime_before_charttime"), 1).otherwise(0)).alias("dq_storetime_before_charttime"),
    F.sum(F.when(F.col("dq_hadm_not_in_admissions"), 1).otherwise(0)).alias("dq_hadm_not_in_admissions"),
    F.sum(F.when(F.col("value_comparator").isNotNull(), 1).otherwise(0)).alias("censored_values"),
    F.sum(F.when(F.col("value_censored_num").isNotNull() & F.col("valuenum").isNotNull(), 1).otherwise(0)).alias("censored_with_valuenum"),
).first().asDict()
bronze_rows = labevents_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_key"] == 0, f"{KEY} contains NULLs"
# labevent_id is the documented primary key; the profiling cells above must have confirmed it
assert post["distinct_key"] == post["silver_rows"], f"{KEY} is not unique"
assert post["null_processed_at"] == 0
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"
assert post["censored_with_valuenum"] == 0, "value_censored_num set where valuenum exists"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("labevents:OK")